In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from itertools import combinations

# Preprocessing

In [ ]:
data = pd.read_csv('/Users/go202/Desktop/equate_sheet/final_df.csv')
data.loc[data['language_name'] == 'Mandarin Chinese', 'models_n'] = 11443
data.loc[data['iso_code'] == 'unk', 'models_n'] = 2
# China stuff
china_rows = data[data['country_iso_code'] == 'CHN']
excluded_columns = ['avg_latency_ms', 'distance_to_closest_university_km', 'distance_to_second_closest_university_km']
numeric_columns = data.select_dtypes(include=[np.number]).columns
columns_to_update = [col for col in numeric_columns if col not in excluded_columns]
china_max_values = china_rows[columns_to_update].max()
mandarin_mask = data['language_name'] == 'Mandarin Chinese'
for col in columns_to_update:
    data.loc[mandarin_mask, col] = china_max_values[col]

#non china stuff
def cc_to_numeric(value):
    if isinstance(value, (int, float)):
        return value

    value = value.strip()
    if 'K' in value:
        return float(value.replace('K', '').strip()) * 1e3
    elif 'M' in value:
        return float(value.replace('M', '').strip()) * 1e6
    elif 'G' in value:
        return float(value.replace('G', '').strip()) * 1e9
    elif 'T' in value:
        return float(value.replace('T', '').strip()) * 1e12
    else:
        # Handle plain numeric strings
        try:
            return float(value)
        except ValueError:
            return np.nan  

data['cc_gb'] = data['cc_gb'].apply(cc_to_numeric)

def wiki_to_numeric(value):
    if isinstance(value, int):
        return value
    if pd.isna(value):
        return 0  
    return int(value.replace(',', ''))


data['wiki_articles_n'] = data['wiki_articles_n'].apply(wiki_to_numeric)

# Weights and Feature Groups Specifications

In [19]:
binary_features = ['bible_exists', 'cybersecurity_law_exists','institutional_status']
log_features = [    'cc_gb',
    'models_n',              
    'opus_gb',
    'archival_gb',
    'xeus_gb', 
    'wikitongues_gb', 'n_speakers', 
    'wiki_active_users'
    ]

In [ ]:
def compute_normalized_weights(features, ranks, group_name=None):
    """
    Computes normalized linear weights from ranks.

    Args:
        features (list): List of feature names.
        ranks (list): Corresponding rank values.
        group_name (str, optional): Name of group for display/debug.

    Returns:
        pd.DataFrame: DataFrame with 'feature', 'rank', 'linear_weight', 'normalized_weight'
    """
    n = len(ranks)
    linear_weights = [1 - (rank - 1) / (n - 1) for rank in ranks]
    total = sum(linear_weights)
    normalized_weights = [w / total for w in linear_weights]

    df = pd.DataFrame({
        "feature": features,
        "rank": ranks,
        "linear_weight": linear_weights,
        "normalized_weight": normalized_weights
    })

    if group_name:
        df["group"] = group_name  

    return df


# Define groups and ranks in a dictionary
feature_groups = {
    "technical": {
        "features": ['cc_gb', 'models_n', 'opus_gb', 'archival_gb', 'bible_exists', 'xeus_gb', 'extra_feature'],  # example 7 features
        "ranks": [1, 2, 3, 4, 5, 6, 7]
    },
    "internet": {
        "features": ['households_with_phones_pct', 'individuals_using_internet_pct',
                     'avg_download_speed_kbps', 'households_with_home_internet_pct',
                     'cybersecurity_law_exists'],
        "ranks": [1, 2, 3, 4, 5]
    },
    "language": {
        "features": ['institutional_status', 'n_speakers', 'wiki_active_users', 'literacy_rate',
                     'admin_1_gdp', 'rnd_gdp_pct', 'admin_1_hdi', 'admin_1_education_level'],
        "ranks": [1, 2, 3, 4, 5, 6, 7, 8]
    }
}

feature_group_definitions = {
    group: values["features"]
    for group, values in feature_groups.items()
}
# Compute weights for each group
all_feature_weights = []
for group_name, group_data in feature_groups.items():
    df_weights = compute_normalized_weights(
        features=group_data["features"],
        ranks=group_data["ranks"],
        group_name=group_name
    )
    all_feature_weights.append(df_weights)


final_feature_weights_df = pd.concat(all_feature_weights, ignore_index=True)


print(final_feature_weights_df)

                              feature  rank  linear_weight  normalized_weight  \
0                               cc_gb     1       1.000000           0.285714   
1                            models_n     2       0.833333           0.238095   
2                             opus_gb     3       0.666667           0.190476   
3                         archival_gb     4       0.500000           0.142857   
4                        bible_exists     5       0.333333           0.095238   
5                             xeus_gb     6       0.166667           0.047619   
6                       extra_feature     7       0.000000           0.000000   
7          households_with_phones_pct     1       1.000000           0.400000   
8      individuals_using_internet_pct     2       0.750000           0.300000   
9             avg_download_speed_kbps     3       0.500000           0.200000   
10  households_with_home_internet_pct     4       0.250000           0.100000   
11           cybersecurity_l

In [20]:
def merge_correlated_features(df, feature_group_definitions, final_feature_weights_df, threshold=0.85):
    merged_df = df.copy()
    updated_group_defs = {}
    updated_weights = final_feature_weights_df.copy()

    for group_name, features in feature_group_definitions.items():
        valid_features = [f for f in features if f in df.columns]
        if len(valid_features) < 2:
            updated_group_defs[group_name] = valid_features
            continue

        corr_matrix = df[valid_features].corr(method='pearson').abs()
        merged = set()
        merged_features = []

        for f1, f2 in combinations(valid_features, 2):
            if corr_matrix.loc[f1, f2] > threshold:
                if f1 not in merged and f2 not in merged:
                    w1 = final_feature_weights_df.loc[final_feature_weights_df['feature'] == f1, 'normalized_weight'].values[0]
                    w2 = final_feature_weights_df.loc[final_feature_weights_df['feature'] == f2, 'normalized_weight'].values[0]
                    merged_name = f"{f1}_{f2}_merged"
                    merged_df[merged_name] = (df[f1]*w1 + df[f2]*w2) / (w1 + w2)

                    merged_features.append(merged_name)
                    updated_weights = updated_weights[~updated_weights['feature'].isin([f1, f2])]
                    updated_weights = pd.concat([
                        updated_weights,
                        pd.DataFrame({'feature': [merged_name], 'normalized_weight': [w1 + w2]})
                    ])
                    merged.update([f1, f2])

        unmerged_features = [f for f in valid_features if f not in merged]
        updated_group_defs[group_name] = unmerged_features + merged_features

    return merged_df, updated_group_defs, updated_weights

In [ ]:
merged_data, updated_group_defs, updated_weights = merge_correlated_features(
    df=data,
    feature_group_definitions=feature_group_definitions,
    final_feature_weights_df=final_feature_weights_df,
    threshold=0.85  
)

In [22]:
updated_group_defs

{'technical': ['opus_gb',
  'archival_gb',
  'bible_exists',
  'xeus_gb',
  'cc_gb_models_n_merged'],
 'internet': ['avg_download_speed_kbps',
  'households_with_home_internet_pct',
  'cybersecurity_law_exists',
  'households_with_phones_pct_individuals_using_internet_pct_merged'],
 'language': ['institutional_status',
  'n_speakers',
  'wiki_active_users',
  'literacy_rate',
  'admin_1_gdp',
  'rnd_gdp_pct',
  'admin_1_hdi_admin_1_education_level_merged']}

# Compute full index 

In [ ]:
def adjusted_geometric_weighted_aggregation_individual_weights(
    df,
    final_feature_weights_df,
    feature_group_definitions,
    binary_features,
    log_scale_features=None,
    language_id_column='language_name', 
    epsilon=1e-6
):
    """
    Calculates an aggregated readiness index using weighted geometric aggregation
    with adjustments for correlated features, log scaling, and binary penalties.

    Args:
        df (pd.DataFrame): The input DataFrame containing feature values for languages.
        final_feature_weights_df (pd.DataFrame): DataFrame with 'feature' and 'normalized_weight' columns.
        feature_group_definitions (dict): A dictionary mapping group names to lists of features in that group.
        binary_features (list): A list of feature names that are binary (0 or 1).
        log_scale_features (list, optional): A list of continuous feature names to apply log scaling to. Defaults to None.
        language_id_column (str, optional): The name of the column containing language identifiers. Not directly used in calculation.
        epsilon (float, optional): Small constant added before log-scaling to prevent log(0). Defaults to 1e-6.

    Returns:
        pd.Series: A pandas Series of aggregated readiness scores, sorted in descending order.
    """

    # Merge correlated features 
    df, feature_group_definitions, final_feature_weights_df = merge_correlated_features(
        df, feature_group_definitions, final_feature_weights_df
    )

    # Replace zero weights with small positive value (if they exist)
    final_feature_weights_df['normalized_weight'] = final_feature_weights_df['normalized_weight'].replace(0, 0.01)
    feature_weight_map = final_feature_weights_df.set_index('feature')['normalized_weight'].to_dict()

    num_groups = len(feature_group_definitions)
    df_processed = df.copy()

    # Identify all non-binary numeric features that will be processed
    non_binary_numeric_features = []
    for feature in df_processed.columns:
        if feature not in binary_features and pd.api.types.is_numeric_dtype(df_processed[feature]):
            non_binary_numeric_features.append(feature)

    # Shifting Continuous Features: Add 1 to all non-binary numeric features to avoid zero values
    for feature in non_binary_numeric_features:
        df_processed[feature] = df_processed[feature].fillna(0) + 1 

    # Log-Transform 
    if log_scale_features:
        for feature in log_scale_features:
            if feature in df_processed.columns and feature in non_binary_numeric_features: # Ensure it's a non-binary numeric feature
                df_processed[feature] = np.log(df_processed[feature] + epsilon)
            else:
                print(f"Warning: Log scale feature '{feature}' not found or is binary/non-numeric. Skipping log transform.")

    # Normalize all non-binary features to a [0, 1] range (Min-Max Scaling)
    for feature in non_binary_numeric_features:
        min_val = df_processed[feature].min()
        max_val = df_processed[feature].max()

        if max_val == min_val: # Avoid division by zero if all values are the same
            df_processed[feature] = 0.5 # Assign a neutral value, or 1 if presence is always positive
        else:
            df_processed[feature] = (df_processed[feature] - min_val) / (max_val - min_val)
            df_processed[feature] = df_processed[feature].apply(lambda x: max(x, epsilon))
   

    aggregated_values = []

    for idx, row in df_processed.iterrows():
        group_multiplicative_factor = 1.0

        for group_name, features_in_group in feature_group_definitions.items():
            group_geometric_product_sum_weighted = 0.0 # Will accumulate log-transformed weighted sum
            group_weight_sum = 0.0
            valid_features_in_group = 0

            # Calculate sum of log(val) * weight for the group
            for feature in features_in_group:
                if feature in df_processed.columns and feature in feature_weight_map and feature not in binary_features:
                    val = row[feature]
                    weight = feature_weight_map[feature]
                    group_geometric_product_sum_weighted += np.log(val) * weight
                    group_weight_sum += weight
                    valid_features_in_group += 1

            if valid_features_in_group > 0:
                if abs(group_weight_sum - 1.0) > 1e-6:
                    # Renormalize weights and apply geometric mean: exp(sum(renormalized_weight * log(val)))
                    renormalized_log_sum = 0.0
                    for feature in features_in_group:
                        if feature in df_processed.columns and feature in feature_weight_map and feature not in binary_features:
                            original_weight = feature_weight_map[feature]
                            renormalized_weight = original_weight / group_weight_sum
                            renormalized_log_sum += np.log(row[feature]) * renormalized_weight
                    group_multiplicative_factor *= np.exp(renormalized_log_sum)
                else:
                    # If weights already sum to 1, apply geometric mean directly
                    group_multiplicative_factor *= np.exp(group_geometric_product_sum_weighted)
            # If no valid features in group, group_multiplicative_factor remains 1.0, effectively no contribution.

        # Binary Feature Adjustment
        binary_penalty = 1.0
        for binary_feature in binary_features:
            if binary_feature in df_processed.columns and binary_feature in feature_weight_map:
                weight = feature_weight_map[binary_feature]
                val = row[binary_feature]

                penalty_factor = 1 - weight * (1 - val)
                # Ensure penalty factor is not negative (if weight > 1) or leads to division by zero later
                binary_penalty *= max(0, penalty_factor) # Ensure non-negative penalty
            else:
                print(f"Warning: Binary feature '{binary_feature}' not found in data or weights. Assuming no penalty.")

        # Final Index Score
    
        final_score = (group_multiplicative_factor ** (1 / num_groups)) * binary_penalty
        aggregated_values.append(final_score)

    return pd.Series(aggregated_values, index=df.index).sort_values(ascending=False)

In [ ]:
aggregated_result = adjusted_geometric_weighted_aggregation_individual_weights(
    df=merged_data,
    final_feature_weights_df=updated_weights,
    feature_group_definitions=updated_group_defs,
    binary_features=binary_features,
    log_scale_features=log_features,
    language_id_column='language_name'  
)

print(aggregated_result)

1705    0.748709
1611    0.499661
2755    0.372975
937     0.369301
2728    0.345428
          ...   
5196    0.000019
5191    0.000017
5195    0.000009
5194    0.000008
5193    0.000008
Length: 6003, dtype: float64


# Genereal Index Frame

In [26]:
aggregated_df = aggregated_result.rename("aggregated_score").to_frame()

# Merge on the index
complete_index = aggregated_df.merge(data, left_index=True, right_index=True, how='inner')